# Subseasonal to Seasonal (S2S) – Neptune, Persistence and Climatology

This notebook provides tools for the analysis and comparison of subseasonal to seasonal (S2S) forecasts using machine learning emulators. Key features include:

- **Loading and visualization** of S2S forecast and reanalysis datasets.
- **Calculation and comparison** of evaluation metrics (e.g., RMSE, ACC, CRPS) for different models and observations.
- **Temporal and spatial** analysis of model performance.
- **Visualization of maps**, time series, and statistical distributions.
- **Support for multiple datasets** and atmospheric/oceanic variables.
- **Interactive exploration** of results for model validation and benchmarking.

This notebook is designed to facilitate model comparison, forecast validation, and in-depth performance analysis over weekly to seasonal timescales.

In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import warnings
import munch
import toml
import sys
import os
warnings.filterwarnings('ignore')
sys.path.append('library')
from atmocean.metrics import compute_and_store_metrics, get_persistence_ds
from atmocean.utils import clean_forecasts, remove_ocean_variables_suffix
from atmocean.macros import ALL_DRIVERS, MODEL_NAMES
from atmocean.visualize import *

os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
forecast_horizon = 60

start_date = '01-01'
end_date = '12-31'
years = [2019,2020,2021]

lead_times = [0, 10, 20, 30, 40, 50, 60]

if 'ocean' in selected_model:
    fsystem = 'ocean'
elif 'atmos' in selected_model:
    fsystem = 'atmos'
else:
    raise ValueError('Model not Recognized')

main_dir = 'PATH-TO-MAIN-DIRECTORY'
data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')
metrics_src = os.path.join(data_dir, 'evaluation_metrics')

persistence_src = os.path.join(forecast_dir, f'persistence-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
climatology_src = os.path.join(forecast_dir, f'climatology-from_1993_to_2017.zarr')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/statistical_evaluation')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/statistical_evaluation')

if '_025' in selected_model:
    persistence_src = persistence_src+'-_025'
    climatology_src = climatology_src+'-_025'

os.makedirs(out_img_dir, exist_ok=True)

plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

n_workers = 4
memory_limit = 948 // n_workers
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=1,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# get all the dates
dates = pd.date_range(f'{years[0]}-{start_date}', f'{years[-1]}-{end_date}', freq='1D')
year = f'{dates[0].year}'
sdate = f'{dates[0].year}-{dates[0].month:02d}-{dates[0].day:02d}'
edate = f'{dates[-1].year}-{dates[-1].month:02d}-{dates[-1].day:02d}'
sdate, edate

In [ ]:
fields_info = munch.munchify(toml.load(os.path.join(data_dir, 'specs/ocean_fields.toml')))
fields_info

In [ ]:
season = 'annual'

if ensemble_size is None:
    forecast_acc_src = os.path.join(metrics_src, f'acc', f'{selected_model}-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
else:
    forecast_acc_src = os.path.join(metrics_src, f'acc', f'{selected_model}-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}-ENS-{ensemble_size:02d}.zarr')

persistence_acc_src = os.path.join(metrics_src, f'acc', f'persistence-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
climatology_acc_src = os.path.join(metrics_src, f'acc', f'climatology-{season}-from_1993_to_2017.zarr')

In [ ]:
# open forecasts
forecasts_ds = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)

# open the climatology dataset
climatology_ds = xr.open_zarr(climatology_src, chunks={})

# open the persistence dataset
persistence_ds = get_persistence_ds(persistence_src, forecasts_ds.isel(ensemble_member=0, missing_dims='ignore'))


In [ ]:
climatology_ds['depth'] = forecasts_ds.depth.values

In [ ]:
# realign the prediction_timedelta range
forecasts_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)
climatology_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)
persistence_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)

In [ ]:
forecasts_ds = forecasts_ds.isel(prediction_timedelta=lead_times)
climatology_ds = climatology_ds.isel(prediction_timedelta=lead_times)
persistence_ds = persistence_ds.isel(prediction_timedelta=lead_times)

In [ ]:
# use only one trajectory as it is intractable to compute metrics with ensemble
if ensemble_size is not None:
    forecasts_ds = forecasts_ds.mean(dim='ensemble_member')

In [ ]:
# get only the drivers present in the dataset
all_fields = list(forecasts_ds.data_vars.keys())
drivers = ALL_DRIVERS
drivers['output']['atmos']['fields_3d'] = [field for field in drivers['output']['atmos']['fields_3d'] if field in all_fields]
drivers['output']['atmos']['fields_2d'] = [field for field in drivers['output']['atmos']['fields_2d'] if field in all_fields]
drivers['output']['ocean']['fields_3d'] = [field for field in drivers['output']['ocean']['fields_3d'] if field in all_fields]
drivers['output']['ocean']['fields_2d'] = [field for field in drivers['output']['ocean']['fields_2d'] if field in all_fields]

In [ ]:
def create_yearly_climatology(climatology_ds: xr.Dataset, years):
    clm = []
    for year in years:
        dates = pd.date_range(f'{year}-01-01', f'{year}-12-31', freq='1D')
        tmp = climatology_ds.copy()
        if len(dates) == 365:
            tmp = tmp.sel(time=~((tmp.time.dt.month == 2) & (tmp.time.dt.day == 29)))
        tmp['time'] = dates
        clm.append(tmp)
    clms = xr.concat(clm, dim='time')
    return clms

In [ ]:
# align climatology with the ground truth coming from 
# the persistence, as it contains the real values of 
# ground truth at `prediction_timedelta` = 0
climatology_ds = create_yearly_climatology(climatology_ds, years)
climatology_ds = xr.concat([
    persistence_ds.isel(prediction_timedelta=slice(0,1)), # correct ground truth
    climatology_ds.isel(prediction_timedelta=slice(1,None))
    ], dim='prediction_timedelta'
)

In [ ]:
if not os.path.exists(forecast_acc_src):
    forecasts_ds = client.persist(forecasts_ds)
    progress(forecasts_ds)

In [ ]:
if not os.path.exists(forecast_acc_src):
    climatology_ds = client.persist(climatology_ds)
    progress(climatology_ds)

In [ ]:
# compute the bias and store it to zarr
forecasts_acc = compute_and_store_metrics(forecasts_ds, lead_times, forecast_acc_src, 'acc', climatology_ds.isel(prediction_timedelta=1))
persistence_acc = compute_and_store_metrics(persistence_ds, lead_times, persistence_acc_src, 'acc', climatology_ds.isel(prediction_timedelta=1))

In [ ]:
if not os.path.exists(forecast_acc_src):
    forecasts_acc = client.compute(forecasts_acc)
    progress(forecasts_acc)

In [ ]:
forecasts_acc = compute_and_store_metrics(forecasts_ds, lead_times, forecast_acc_src, 'acc', climatology_ds.isel(prediction_timedelta=1))

In [ ]:
# materializza subito: sono piccoli (scorecard) ma vengono riletti/plottati molte volte nelle celle successive
forecasts_acc = forecasts_acc.load()
persistence_acc = persistence_acc.load()

In [ ]:
# get the fields
fields_3d_ = drivers['output']['atmos']['fields_3d'] + drivers['output']['ocean']['fields_3d']
fields_2d_ = drivers['output']['atmos']['fields_2d'] + drivers['output']['ocean']['fields_2d']
# remove the suffix
fields_3d, fields_2d = [], []
for field in fields_3d_:
    field_rename = '_'.join(field.split('_')[:-1])
    fields_3d.append(field_rename)
for field in fields_2d_:
    field_rename = '_'.join(field.split('_')[:-1])
    fields_2d.append(field_rename)
# add ocean fields
ice_fields_2d = ['siconc', 'sithick']
other_fields_2d = [field for field in fields_2d if field not in ice_fields_2d]
# rename fields info keys
fields_info_new = fields_info.copy()
for key,value in fields_info.items():
    new_key = '_'.join(key.split('_')[:-1])
    fields_info_new[new_key] = value
    del fields_info_new[key]
fields_info = fields_info_new.copy()

In [ ]:
persistence_acc['prediction_timedelta'] = pd.to_timedelta(persistence_acc['prediction_timedelta'].values, unit='days')

In [ ]:
metrics_dict = {
    'acc': {
        'neptune': remove_ocean_variables_suffix(forecasts_acc), 
        'pers': remove_ocean_variables_suffix(persistence_acc), 
    }
}

In [ ]:
model_names = {
    'neptune': 'Neptune', 
    'pers': 'Persistence', 
    'clm': 'Climatology', 
}

In [ ]:
def parse_ids(ids: str):
    ri, ci = ids.split(',')
    if ':' in ri:
        ri = ri.split(':')
        ri = slice(int(ri[0]), int(ri[1]))
    else: ri = int(ri)
    if ':' in ci:
        ci = ci.split(':')
        ci = slice(int(ci[0]), int(ci[1]))
    else: ci = int(ci)
    return ri, ci

In [ ]:
fields_info['mlotst']['long_name'] = 'Mixed Layer Depth'

In [ ]:
metric_name = 'acc'
grid_spec_args = dict(num_figures = (2, 6), positions = [
    '0,0:2', '0,2:4', '0,4:6', '1,0:2', '1,2:4', '1,4:6',
])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
fields = fields_3d + fields_2d[:2]

fig, axes = plt.subplots(1, 1, figsize=(12,6))
axes.axis('off')

# initialize grid spec
gs = GridSpec(*grid_spec_args['num_figures'], figure=fig)
axs = []

for i,field in tqdm(enumerate(fields), total=len(fields)):
    metric = metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=-1, depth=0, level=0, missing_dims='ignore')[field].values
    cmap, norm, levels = get_sst_bias_colormap(-1.0, 1.0)
    grid_pos = grid_spec_args['positions'][i]
    im, ax = plot_spatial_metric_on_globe(
        metric=metric, grid=gs[parse_ids(grid_pos)], title=fields_info[field]['long_name'].capitalize(),
        fig=fig, lons=lons, lats=lats, central_longitude=180,
        cmap=cmap, norm=norm, levels=levels, left_labels=True if (i==0 or i==3) else False, bottom_labels=True
    )
    axs.append(ax)
    # single colorbar
    cbar_label = fields_info[field]["units"]
    if cbar_label == 'degrees_C': cbar_label = '°C'
    kwargs = dict(
        ax=ax, orientation='horizontal', shrink=0.9,
    )
    kwargs['label'] = cbar_label
    kwargs['ticks'] = np.linspace(-1.0, 1.0, 5)
    cbar = fig.colorbar(im, **kwargs)
# adjust the plot
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=-0.1, hspace=0.3)
outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_lead_time_60_two_rows.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
metric_name = 'acc'
grid_spec_args = dict(num_figures = (1, 12), positions = [
    '0,0:2', '0,2:4', '0,4:6', '0,6:8', '0,8:10', '0,10:12',
])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
fields = fields_3d + fields_2d[:2]

fig, axes = plt.subplots(1, 1, figsize=(24,3))
fig.suptitle(f'{metric_name.upper()} at t+60 lead time', fontsize=16, y=1.1)
axes.axis('off')

# initialize grid spec
gs = GridSpec(*grid_spec_args['num_figures'], figure=fig)
axs = []

for i,field in tqdm(enumerate(fields), total=len(fields)):
    metric = metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=-1, depth=0, level=0, missing_dims='ignore')[field].values
    cmap, norm, levels = get_sst_bias_colormap(-1.0, 1.0)
    # cmap, norm, levels = get_colormap_from_cmap('RdBu_r', vmin=-1.0, vmax=1.0)
    grid_pos = grid_spec_args['positions'][i]
    im, ax = plot_spatial_metric_on_globe(
        metric=metric, grid=gs[parse_ids(grid_pos)], title=fields_info[field]['long_name'].capitalize(),
        fig=fig, lons=lons, lats=lats, central_longitude=180,
        cmap=cmap, norm=norm, levels=levels, left_labels = True if i == 0 else False, bottom_labels = True
    )
    axs.append(ax)
    # single colorbar
    kwargs = dict(
        ax=ax, orientation='horizontal',
        shrink=0.9, # aspect=60,
        # cax=cax, pad=0.05, aspect=60, shrink=0.9, format=ScalarFormatter()
    )
    # kwargs['label'] = cbar_label
    kwargs['ticks'] = np.linspace(-1.0, 1.0, 5)
    cbar = fig.colorbar(im, **kwargs)
# adjust the plot
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=0.0, hspace=0.5)
outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_lead_time_60.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()
# plt.show()

# Anomaly Correlation Coefficient

In [ ]:
metric_name = 'acc'
cmap, norm, levels = get_sst_bias_colormap(-1.0, 1.0)

In [ ]:
grid_spec_args = dict(num_figures = (2, 6), positions = ['0,0:2', '0,2:4', '0,4:6', '1,0:2', '1,2:4', '1,4:6'])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
for field in tqdm(fields_3d + fields_2d):
    outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_{field}.png') if save_figures else None
    metric_values = [
        metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=i+1, depth=0, level=0, missing_dims='ignore')[field].values
        for i,t in enumerate(lead_times[1:])
    ]
    cmap, norm, levels = get_sst_bias_colormap(-1.0, 1.0)
    lead_time_labels = [f't+{t}' for t in lead_times[1:]]
    plot_metrics_on_robinson_globe(
        metric_values=metric_values, 
        model_names=lead_time_labels, 
        grid_spec_args=grid_spec_args, 
        central_longitude=180, 
        metric_name=metric_name, 
        field=field, 
        fields_info=fields_info, 
        lats=lats, 
        lons=lons, 
        cmap=cmap, 
        norm=norm, 
        levels=levels, 
        start_year=sdate, 
        end_year=edate, 
        outfile=outfile, 
        use_units=True, 
    )

In [ ]:
polar = 'north'

model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=1e-2, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=model1, 
    polar=polar, 
    cmap=cmap, 
    divide_by=1, 
    dext=[-1.0, 1.0], 
    outfile=outfile
)

# model1 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
# polar_plot_ice_fields(
#     ds=metrics_dict[metric_name][model1], 
#     lead_times=lead_times[1:], 
#     fields=ice_fields_2d, 
#     fields_info=fields_info, 
#     remove_below_x=1e-2, 
#     sdate=sdate, 
#     edate=edate, 
#     metric_name=metric_name, 
#     model_name=model1, 
#     polar=polar, 
#     cmap=cmap, 
#     divide_by=1, 
#     dext=[-1.0, 1.0], 
#     outfile=outfile
# )

# NOTE: ITS MEANINGLESS TO SUBTRACT ACCs
# model1 = 'neptune'
# model2 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
# polar_plot_ice_fields(
#     ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
#     lead_times=lead_times[1:], 
#     fields=ice_fields_2d, 
#     fields_info=fields_info, 
#     remove_below_x=None, 
#     sdate=sdate, 
#     edate=edate, 
#     metric_name=metric_name, 
#     model_name=f'({model_names[model1]} - {model_names[model2]})', 
#     polar=polar, 
#     cmap=cmap, 
#     divide_by=2, 
#     outfile=outfile
# )

In [ ]:
polar = 'south'

model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=1e-2, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=model1, 
    polar=polar, 
    cmap=cmap, 
    divide_by=1, 
    dext=[-1.0, 1.0], 
    outfile=outfile
)

# model1 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
# polar_plot_ice_fields(
#     ds=metrics_dict[metric_name][model1], 
#     lead_times=lead_times[1:], 
#     fields=ice_fields_2d, 
#     fields_info=fields_info, 
#     remove_below_x=1e-2, 
#     sdate=sdate, 
#     edate=edate, 
#     metric_name=metric_name, 
#     model_name=model1, 
#     polar=polar, 
#     cmap=cmap, 
#     divide_by=1, 
#     dext=[-1.0, 1.0], 
#     outfile=outfile
# )

# NOTE: ITS MEANINGLESS TO SUBTRACT ACCs
# model1 = 'neptune'
# model2 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
# polar_plot_ice_fields(
#     ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
#     lead_times=lead_times[1:], 
#     fields=ice_fields_2d, 
#     fields_info=fields_info, 
#     remove_below_x=None, 
#     sdate=sdate, 
#     edate=edate, 
#     metric_name=metric_name, 
#     model_name=f'({model_names[model1]} - {model_names[model2]})', 
#     polar=polar, 
#     cmap=cmap, 
#     divide_by=2, 
#     outfile=outfile
# )

In [ ]:
outfile = os.path.join(out_img_dir, f'{metric_name}_surface_all_models_compared.png') if save_figures else None
metrics_list = [
    metrics_dict[metric_name]['neptune'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['pers'].mean(dim=['latitude','longitude']), 
]
labels = [
    model_names['neptune'], 
    model_names['pers'], 
]
colors = [
    'red', 
    'orange', 
]
surface_plots_multiple_models(
    metrics_list=metrics_list, 
    labels=labels, 
    colors=colors, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    sdate=sdate, 
    edate=edate, 
    fields_info=fields_info, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    metric_name=metric_name, 
    outfile=outfile, 
)

In [ ]:
outfile = os.path.join(out_img_dir, f'{metric_name}_vertical_profile_all_models_compared.png') if save_figures else None
metrics_list = [
    metrics_dict[metric_name]['neptune'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['pers'].mean(dim=['latitude','longitude']), 
]
labels = [
    model_names['neptune'], 
    model_names['pers'], 
]
colors = [
    'red', 
    'orange', 
]
vertical_plots_multiple_models(
    metrics_list=metrics_list, 
    labels=labels, 
    colors=colors, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    vert_levels=drivers['depths'], 
    fields_info=fields_info, 
    sdate=sdate, 
    edate=edate, 
    fields_3d=fields_3d, 
    metric_name=metric_name, 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_scorecard_{model1}.png') if save_figures else None
metrics_scorecard_by_lead_times(
    metrics_ds=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    model=model_names[model1], 
    cmap=cmap, 
    dvmin=-1.0, 
    dvmax=1.0, 
    outfile=outfile, 
)

In [ ]:
model1 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_scorecard_{model1}.png') if save_figures else None
metrics_scorecard_by_lead_times(
    metrics_ds=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    model=model_names[model1], 
    cmap=cmap, 
    dvmin=-1.0, 
    dvmax=1.0, 
    outfile=outfile, 
)

In [ ]:
# NOTE: ITS MEANINGLESS TO SUBTRACT ACCs
# model1 = 'neptune'
# model2 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_difference_{model1}_vs_{model2}.png') if save_figures else None
# metrics_difference_scorecard_by_lead_times(
#     metrics_model1=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
#     metrics_model2=metrics_dict[metric_name][model2].mean(dim=['latitude','longitude']), 
#     metric_name=metric_name, 
#     fields_info=fields_info, 
#     fsystem=fsystem, 
#     sdate=sdate, 
#     edate=edate, 
#     fields_3d=fields_3d, 
#     fields_2d=fields_2d, 
#     vert_levels=drivers['depths'], 
#     lead_times_labels=lead_times, 
#     lead_times=lead_times, 
#     model1=model_names[model1], 
#     model2=model_names[model2], 
#     cmap=cmap, 
#     normalize=False, 
#     outfile=outfile, 
# )

In [ ]:
model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_spatial.png') if save_figures else None
plot_spatial_metric(
    metric_spatial=metrics_dict[metric_name][model1], 
    metric_name=metric_name, 
    model=model_names[model1], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    fields_info=fields_info, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    sdate=sdate, 
    edate=edate, 
    outfile=outfile, 
    divide_by=1, 
    dvmin=-1, 
    dvmax=1, 
    cmap=cmap, 
)

In [ ]:
# model1 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_spatial.png') if save_figures else None
# plot_spatial_metric(
#     metric_spatial=metrics_dict[metric_name][model1], 
#     metric_name=metric_name, 
#     model=model_names[model1], 
#     fields_3d=fields_3d, 
#     fields_2d=fields_2d, 
#     fields_info=fields_info, 
#     lead_times=[i+1 for i in range(len(lead_times[1:]))], 
#     lead_times_labels=lead_times[1:], 
#     sdate=sdate, 
#     edate=edate, 
#     outfile=outfile, 
#     divide_by=1, 
#     dvmin=-1, 
#     dvmax=1, 
#     cmap=cmap, 
# )

In [ ]:
# model1 = 'neptune'
# model2 = 'pers'
# outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}-{model2}_spatial_difference.png') if save_figures else None
# plot_spatial_metrics_difference(
#     metric_model1=metrics_dict[metric_name][model1], 
#     metric_model2=metrics_dict[metric_name][model2], 
#     metric_name=metric_name, 
#     fields_3d=fields_3d, 
#     fields_2d=fields_2d, 
#     fields_info=fields_info, 
#     lead_times_ids=lead_times, 
#     lead_times_labels=lead_times, 
#     model1=model_names[model1], 
#     model2=model_names[model2], 
#     sdate=sdate, 
#     edate=edate, 
#     cmap=cmap, 
#     dext=1, 
#     outfile=outfile, 
# )